## Section 1: Load and Explore the Datasets

Import necessary libraries and load the data from CSV files.

In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

deliveries = pd.read_csv('deliveries.csv')
matches = pd.read_csv('matches.csv')

print("Deliveries Dataset:")
print(f"Shape: {deliveries.shape}")
print(f"\nFirst few rows:")
print(deliveries.head())
print(f"\nData types:")
print(deliveries.dtypes)
print(f"\nBasic statistics:")
print(deliveries.describe())

Deliveries Dataset:
Shape: (260920, 17)

First few rows:
   match_id  inning           batting_team                 bowling_team  over  \
0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   

   ball       batter   bowler  non_striker  batsman_runs  extra_runs  \
0     1   SC Ganguly  P Kumar  BB McCullum             0           1   
1     2  BB McCullum  P Kumar   SC Ganguly             0           0   
2     3  BB McCullum  P Kumar   SC Ganguly             0           1   
3     4  BB McCullum  P Kumar   SC Ganguly             0           0   
4     5  BB McCullum  P Kumar   SC Ganguly             0           0   

   total_runs extras_ty

In [53]:

data = deliveries.merge(matches, left_on='match_id', right_on='id', how='left')
data['wicket'] = data['player_dismissed'].notna().astype(int)
data['date'] = pd.to_datetime(data['date'], errors='coerce')

In [54]:

print("Matches Dataset:")
print(f"Shape: {matches.shape}")
print(f"\nFirst few rows:")
print(matches.head())
print(f"\nData types:")
print(matches.dtypes)
print(f"\nColumns: {matches.columns.tolist()}")

Matches Dataset:
Shape: (1095, 20)

First few rows:
       id   season        city        date match_type player_of_match  \
0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   

                                        venue                        team1  \
0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
2                            Feroz Shah Kotla             Delhi Daredevils   
3                            Wankhede Stadium               Mumbai Indians   
4                                Eden Gardens        Kolkata Knight Riders   

                         team2          

## Section 2: Data Cleaning and Preparation

Merge datasets and prepare data for wicket analysis.

In [55]:

print(f"\n\nMissing values in deliveries:")
print(deliveries.isnull().sum())



Missing values in deliveries:
match_id                 0
inning                   0
batting_team             0
bowling_team             0
over                     0
ball                     0
batter                   0
bowler                   0
non_striker              0
batsman_runs             0
extra_runs               0
total_runs               0
extras_type         246795
is_wicket                0
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64


In [56]:

print("ORIGINAL UNIQUE STADIUMS:")
unique_stadiums_before = sorted(data['venue'].unique())
print(f"\nTotal unique stadiums: {len(unique_stadiums_before)}\n")
for i, stadium in enumerate(unique_stadiums_before, 1):
    count = len(data[data['venue'] == stadium])
    print(f"{i}. {stadium} ({count} records)")

stadium_mapping = {
    'Arun Jaitley Stadium': 'Arun Jaitley Stadium',
    'Arun Jaitely Stadium': 'Arun Jaitley Stadium',
    'Feroz Shah Kotla': 'Feroz Shah Kotla',
    'Feroz Shah Kotla, Delhi': 'Feroz Shah Kotla',

    'Wankhede Stadium': 'Wankhede Stadium',
    'Wankhede Stadium, Mumbai': 'Wankhede Stadium',
    'Brabourne Stadium': 'Brabourne Stadium',

    'M.Chinnaswamy Stadium': 'M. Chinnaswamy Stadium',
    'M. Chinnaswamy Stadium': 'M. Chinnaswamy Stadium',
    'M Chinnaswamy Stadium': 'M. Chinnaswamy Stadium',
    
    'Eden Gardens': 'Eden Gardens',
    'Eden Gardens, Kolkata': 'Eden Gardens',
    
    'MA Chidambaram Stadium, Chepauk': 'M. A. Chidambaram Stadium',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'M. A. Chidambaram Stadium',
    'M.A. Chidambaram Stadium': 'M. A. Chidambaram Stadium',
    'M. A. Chidambaram Stadium': 'M. A. Chidambaram Stadium',
    
    'Rajiv Gandhi International Cricket Stadium': 'Rajiv Gandhi International Stadium, Uppal',
    'Rajiv Gandhi International Cricket Stadium, Hyderabad': 'Rajiv Gandhi International Stadium, Uppal',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium, Uppal',
    
    'Sawai Mansingh Stadium': 'Sawai Mansingh Stadium',
    'Sawai Mansingh Stadium, Jaipur': 'Sawai Mansingh Stadium',
    
    'Punjab Cricket Association Stadium': 'Punjab Cricket Association Stadium, Mohali',
    'Punjab Cricket Association Stadium, Mohali': 'Punjab Cricket Association Stadium, Mohali',
    'Punjab Cricket Association Stadium, Chandigarh': 'Punjab Cricket Association Stadium, Mohali',
    
    'Holkar Cricket Stadium': 'Holkar Cricket Stadium',
    'Holkar Cricket Stadium, Indore': 'Holkar Cricket Stadium',
    
    'Narendra Modi Stadium': 'Narendra Modi Stadium',
    'Narendra Modi Stadium, Ahmedabad': 'Narendra Modi Stadium',
    'HPCA Cricket Stadium': 'HPCA Cricket Stadium',
    'Saurashtra Cricket Association Stadium': 'Saurashtra Cricket Association Stadium',
    'Maharashtra Cricket Association Stadium': 'Maharashtra Cricket Association Stadium',
    'Vidarbha Cricket Association Stadium': 'Vidarbha Cricket Association Stadium',
    'Barsapara Cricket Stadium': 'Barsapara Cricket Stadium',
    'Dr. Y.S. Rajasekhara Reddy Cricket Complex': 'Dr. Y.S. Rajasekhara Reddy Cricket Complex',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Complex': 'Dr. Y.S. Rajasekhara Reddy Cricket Complex',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Complex, Visakhapatnam': 'Dr. Y.S. Rajasekhara Reddy Cricket Complex',
    'Dubai International Cricket Stadium': 'Dubai International Cricket Stadium',
}

for stadium in unique_stadiums_before:
    if stadium not in stadium_mapping:
        stadium_mapping[stadium] = stadium

# Apply mapping to standardize stadium names
data['venue'] = data['venue'].replace(stadium_mapping)

unique_stadiums_after = sorted(data['venue'].unique())
print(f"\nSTANDARDIZED UNIQUE STADIUMS:\
\nTotal unique stadiums after mapping: {len(unique_stadiums_after)}\n")
for i, stadium in enumerate(unique_stadiums_after, 1):
    count = len(data[data['venue'] == stadium])
    print(f"{i}. {stadium} ({count} records)")


ORIGINAL UNIQUE STADIUMS:

Total unique stadiums: 58

1. Arun Jaitley Stadium (3356 records)
2. Arun Jaitley Stadium, Delhi (3963 records)
3. Barabati Stadium (1695 records)
4. Barsapara Cricket Stadium, Guwahati (739 records)
5. Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow (3283 records)
6. Brabourne Stadium (2469 records)
7. Brabourne Stadium, Mumbai (4057 records)
8. Buffalo Park (715 records)
9. De Beers Diamond Oval (726 records)
10. Dr DY Patil Sports Academy (3993 records)
11. Dr DY Patil Sports Academy, Mumbai (4905 records)
12. Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium (3037 records)
13. Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam (500 records)
14. Dubai International Cricket Stadium (11229 records)
15. Eden Gardens (17988 records)
16. Eden Gardens, Kolkata (3858 records)
17. Feroz Shah Kotla (13950 records)
18. Green Park (921 records)
19. Himachal Pradesh Cricket Association Stadium (2159 records)
20. Himachal Pradesh

## Section 3: Wickets Analysis by Stadium

Comprehensive analysis of wickets across different stadiums.

In [57]:
wickets_by_stadium = data[data['wicket'] == 1].groupby('venue').agg({
    'wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()
wickets_by_stadium.columns = ['Stadium', 'Total_Wickets', 'Matches']
wickets_by_stadium['Avg_Wickets_Per_Match'] = (wickets_by_stadium['Total_Wickets'] / 
                                                 wickets_by_stadium['Matches']).round(2)
wickets_by_stadium = wickets_by_stadium.sort_values('Total_Wickets', ascending=False)

print("Wickets Analysis by Stadium:")
print(wickets_by_stadium.to_string())

Wickets Analysis by Stadium:
                                                                  Stadium  Total_Wickets  Matches  Avg_Wickets_Per_Match
51                                                       Wankhede Stadium           1422      118                  12.05
14                                                           Eden Gardens           1071       93                  11.52
23                                              M. A. Chidambaram Stadium            923       76                  12.14
24                                                 M. Chinnaswamy Stadium            922       80                  11.52
15                                                       Feroz Shah Kotla            697       60                  11.62
43                                                 Sawai Mansingh Stadium            623       57                  10.93
39                              Rajiv Gandhi International Stadium, Uppal            565       49                  11.53
13 

In [58]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

top_stadiums = wickets_by_stadium.head(10)

fig.add_trace(
    go.Bar(x=top_stadiums['Stadium'], y=top_stadiums['Total_Wickets'],
           name='Total Wickets', marker_color='indianred'),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(x=top_stadiums['Stadium'], y=top_stadiums['Avg_Wickets_Per_Match'],
               name='Avg Wickets/Match', mode='lines+markers', 
               line=dict(color='blue', width=3), marker=dict(size=8)),
    secondary_y=True,
)

fig.update_layout(
    title_text="Top 10 Stadiums by Wickets (Total and Average)",
    hovermode='x unified',
    height=500,
    template='plotly_white'
)
fig.update_xaxes(title_text="Stadium")
fig.update_yaxes(title_text="Total Wickets", secondary_y=False)
fig.update_yaxes(title_text="Avg Wickets per Match", secondary_y=True)
fig.show()

## Section 4: 1st Innings Wickets Analysis

Analyze wicket patterns specifically in first innings.

In [59]:
first_innings = data[data['inning'] == 1]
wickets_1st_innings = first_innings[first_innings['wicket'] == 1].groupby('venue').agg({
    'wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()
wickets_1st_innings.columns = ['Stadium', 'Total_Wickets_1st', 'Matches_1st']
wickets_1st_innings['Avg_Wickets_1st'] = (wickets_1st_innings['Total_Wickets_1st'] / 
                                           wickets_1st_innings['Matches_1st']).round(2)
wickets_1st_innings = wickets_1st_innings.sort_values('Total_Wickets_1st', ascending=False)

print("1st Innings - Wickets by Stadium (Top 10):")
print(wickets_1st_innings.head(10).to_string())

1st Innings - Wickets by Stadium (Top 10):
                                       Stadium  Total_Wickets_1st  Matches_1st  Avg_Wickets_1st
51                            Wankhede Stadium                725          118             6.14
14                                Eden Gardens                583           93             6.27
24                      M. Chinnaswamy Stadium                490           80             6.12
23                   M. A. Chidambaram Stadium                459           76             6.04
15                            Feroz Shah Kotla                360           60             6.00
43                      Sawai Mansingh Stadium                325           57             5.70
39   Rajiv Gandhi International Stadium, Uppal                312           49             6.37
13         Dubai International Cricket Stadium                265           46             5.76
37  Punjab Cricket Association Stadium, Mohali                223           35             6.

In [60]:
fig = px.bar(
    wickets_1st_innings.head(8), 
    x='Stadium', 
    y='Total_Wickets_1st',
    title='Top 8 Stadiums - 1st Innings Wickets',
    labels={
        'Total_Wickets_1st': 'Total Wickets (1st Innings)',
        'Avg_Wickets_1st': 'Avg Wickets/Match'
    },
    color='Avg_Wickets_1st',
    color_continuous_scale='Viridis'
)

fig.update_layout(height=500, template='plotly_white')

fig.show()

## Section 5: 2nd Innings Wickets Analysis

Analyze wicket patterns specifically in second innings and compare with first innings.

In [61]:
second_innings = data[data['inning'] == 2]
wickets_2nd_innings = second_innings[second_innings['wicket'] == 1].groupby('venue').agg({
    'wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()
wickets_2nd_innings.columns = ['Stadium', 'Total_Wickets_2nd', 'Matches_2nd']
wickets_2nd_innings['Avg_Wickets_2nd'] = (wickets_2nd_innings['Total_Wickets_2nd'] / 
                                           wickets_2nd_innings['Matches_2nd']).round(2)
wickets_2nd_innings = wickets_2nd_innings.sort_values('Total_Wickets_2nd', ascending=False)

print("2nd Innings - Wickets by Stadium (Top 10):")
print(wickets_2nd_innings.head(10).to_string())

2nd Innings - Wickets by Stadium (Top 10):
                                       Stadium  Total_Wickets_2nd  Matches_2nd  Avg_Wickets_2nd
51                            Wankhede Stadium                695          116             5.99
14                                Eden Gardens                488           92             5.30
23                   M. A. Chidambaram Stadium                461           76             6.07
24                      M. Chinnaswamy Stadium                430           77             5.58
15                            Feroz Shah Kotla                337           58             5.81
43                      Sawai Mansingh Stadium                298           56             5.32
13         Dubai International Cricket Stadium                273           45             6.07
39   Rajiv Gandhi International Stadium, Uppal                253           49             5.16
37  Punjab Cricket Association Stadium, Mohali                188           34             5.

In [62]:
fig = px.bar(wickets_2nd_innings.head(8), 
             x='Stadium', y='Total_Wickets_2nd',
             title='Top 8 Stadiums - 2nd Innings Wickets',
             labels={'Total_Wickets_2nd': 'Total Wickets (2nd Innings)'},
             color='Avg_Wickets_2nd',
             color_continuous_scale='Plasma')
fig.update_layout(height=500, template='plotly_white')
fig.show()

## Section 6: Season-wise Wickets Analysis

Analyze wicket trends across different IPL seasons.

In [63]:
wickets_by_season = data[data['wicket'] == 1].groupby('season').agg({
    'wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()
wickets_by_season.columns = ['Season', 'Total_Wickets', 'Matches']
wickets_by_season['Avg_Wickets_Per_Match'] = (wickets_by_season['Total_Wickets'] / 
                                               wickets_by_season['Matches']).round(2)
wickets_by_season = wickets_by_season.sort_values('Season')

print("Wickets by Season:")
print(wickets_by_season.to_string())

season_innings = data[(data['wicket'] == 1) & (data['inning'].isin([1, 2]))].groupby(['season', 'inning']).agg({
    'wicket': 'sum',
}).reset_index()
season_innings.columns = ['Season', 'Inning', 'Wickets']
season_innings_pivot = season_innings.pivot(index='Season', columns='Inning', values='Wickets').fillna(0)
season_innings_pivot.columns = ['1st Innings', '2nd Innings']

print("\n\nWickets by Season and Innings:")
print(season_innings_pivot)

Wickets by Season:
     Season  Total_Wickets  Matches  Avg_Wickets_Per_Match
0   2007/08            690       58                  11.90
1      2009            698       57                  12.25
2   2009/10            725       60                  12.08
3      2011            813       73                  11.14
4      2012            858       74                  11.59
5      2013            912       76                  12.00
6      2014            674       60                  11.23
7      2015            691       59                  11.71
8      2016            666       60                  11.10
9      2017            711       59                  12.05
10     2018            722       60                  12.03
11     2019            685       60                  11.42
12  2020/21            677       60                  11.28
13     2021            717       60                  11.95
14     2022            912       74                  12.32
15     2023            916       74  

In [64]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

top_stadiums = data[data['wicket'] == 1]['venue'].value_counts().head(5).index.tolist()

fig = make_subplots(specs=[[{"secondary_y": True}]])
buttons = []
traces_per_stadium = 2

for i, stadium in enumerate(top_stadiums):
    stadium_data = data[(data['venue'] == stadium) & (data['wicket'] == 1)]
    
    season_stats = data[data['venue'] == stadium].groupby('season').agg(
        Total_Wickets=('wicket', 'sum'),
        Matches=('match_id', 'nunique')
    ).reset_index()
    
    season_stats['Avg_Wickets_Per_Match'] = season_stats.apply(
        lambda row: row['Total_Wickets'] / row['Matches'] if row['Matches'] > 0 else 0, axis=1
    )

    is_visible = (i == 0)

    fig.add_trace(
        go.Bar(
            x=season_stats['season'], y=season_stats['Total_Wickets'],
            name=f'{stadium} Total Wickets', marker_color='lightblue',
            visible=is_visible
        ),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=season_stats['season'], y=season_stats['Avg_Wickets_Per_Match'],
            name=f'{stadium} Avg Wickets/Match', mode='lines+markers',
            line=dict(color='red', width=3), marker=dict(size=8),
            visible=is_visible
        ),
        secondary_y=True,
    )

    visibility = [False] * (len(top_stadiums) * traces_per_stadium)
    visibility[i*traces_per_stadium : (i+1)*traces_per_stadium] = [True, True]
    
    buttons.append(dict(
        label=stadium,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Wickets Trend by Season - {stadium}"}]
    ))

fig.update_layout(
    updatemenus=[dict(
        active=0, buttons=buttons, x=1.3, y=0.51, xanchor='center', 
        direction='down', showactive=True
    )],
    title_text=f"Wickets Trend by Season - {top_stadiums[0]}",
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.update_xaxes(title_text="Season")
fig.update_yaxes(title_text="Total Wickets", secondary_y=False)
fig.update_yaxes(title_text="Avg Wickets per Match", secondary_y=True)

fig.show()

## Section 7: Cumulative Wickets Analysis

Analyze cumulative wicket trends over time.

In [65]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

top_stadiums = data[data['wicket'] == 1]['venue'].value_counts().head(5).index.tolist()

fig = make_subplots(
    rows=1, cols=3,
    specs=[[{'type':'scatter'}, {'type':'bar'}, {'type':'bar'}]],
    subplot_titles=('Cumulative Wickets Over Time', 'Wickets/Day by Season', 'Total Wickets by Season')
)

buttons = []
traces_per_stadium = 3

for i, stadium in enumerate(top_stadiums):
    stadium_data = data[data['venue'] == stadium].copy()
    
    daily_wickets = stadium_data.groupby('date')['wicket'].sum().reset_index()
    daily_wickets['cumulative_wickets'] = daily_wickets['wicket'].cumsum()

    season_stats = stadium_data.groupby('season').agg(
        Total_Wickets=('wicket', 'sum'),
        Days_Played=('date', 'nunique')
    ).reset_index()
    season_stats['Wickets_Per_Day'] = season_stats.apply(
        lambda row: row['Total_Wickets'] / row['Days_Played'] if row['Days_Played'] > 0 else 0, axis=1
    )

    is_visible = (i == 0) 

    fig.add_trace(
        go.Scatter(
            x=daily_wickets['date'], y=daily_wickets['cumulative_wickets'],
            mode='lines', name=f'{stadium} Cumulative',
            line=dict(color='darkblue', width=2), fill='tozeroy',
            hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Cumulative Wickets:</b> %{y}',
            visible=is_visible
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=season_stats['season'], y=season_stats['Wickets_Per_Day'],
            name=f'{stadium} Wickets/Day', marker_color='coral',
            hovertemplate='<b>%{x}</b><br>Wickets/Day: %{y:.2f}', showlegend=False,
            visible=is_visible
        ),
        row=1, col=2
    )

    fig.add_trace(
        go.Bar(
            x=season_stats['season'], y=season_stats['Total_Wickets'],
            name=f'{stadium} Total Wickets', marker_color='steelblue',
            hovertemplate='<b>%{x}</b><br>Wickets: %{y}', showlegend=False,
            visible=is_visible
        ),
        row=1, col=3
    )

    visibility = [False] * (len(top_stadiums) * traces_per_stadium)
    visibility[i*traces_per_stadium : (i+1)*traces_per_stadium] = [True, True, True]
    
    buttons.append(dict(
        label=stadium,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Cumulative Wickets Analysis - {stadium}"}]
    ))

fig.update_layout(
    updatemenus=[dict(
        active=0, buttons=buttons, x=0.5, y=1.2, xanchor='center', 
        direction='down', showactive=True
    )],
    height=550, width=1800,
    title_text=f'Cumulative Wickets Analysis - {top_stadiums[0]}',
    template='plotly_white', showlegend=False
)

fig.update_xaxes(title_text='Date', row=1, col=1, tickangle=45)
fig.update_xaxes(title_text='Season', row=1, col=2, tickangle=45)
fig.update_xaxes(title_text='Season', row=1, col=3, tickangle=45)
fig.update_yaxes(title_text='Cumulative Wickets', row=1, col=1)
fig.update_yaxes(title_text='Wickets/Day', row=1, col=2)
fig.update_yaxes(title_text='Total Wickets', row=1, col=3)

fig.show()



print("TOP 5 STADIUMS - CUMULATIVE WICKET PATTERNS")


stadium_cumulative_stats = []
for stadium in top_stadiums:
    stadium_wickets = data[(data['venue'] == stadium) & (data['wicket'] == 1)]
    total = len(stadium_wickets)
    matches = stadium_wickets['match_id'].nunique()
    
    avg_wickets = round(total / matches, 2) if matches > 0 else 0.00
    stadium_cumulative_stats.append({
        'Stadium': stadium,
        'Total_Wickets': total,
        'Matches': matches,
        'Wickets_Per_Match': avg_wickets
    })
    
    print(f"\n{stadium}:")
    print(f"  Total Wickets: {total}")
    print(f"  Matches: {matches}")
    print(f"  Avg Wickets/Match: {avg_wickets:.2f}")

stadium_cumulative_df = pd.DataFrame(stadium_cumulative_stats)

TOP 5 STADIUMS - CUMULATIVE WICKET PATTERNS

Wankhede Stadium:
  Total Wickets: 1422
  Matches: 118
  Avg Wickets/Match: 12.05

Eden Gardens:
  Total Wickets: 1071
  Matches: 93
  Avg Wickets/Match: 11.52

M. A. Chidambaram Stadium:
  Total Wickets: 923
  Matches: 76
  Avg Wickets/Match: 12.14

M. Chinnaswamy Stadium:
  Total Wickets: 922
  Matches: 80
  Avg Wickets/Match: 11.53

Feroz Shah Kotla:
  Total Wickets: 697
  Matches: 60
  Avg Wickets/Match: 11.62


## Section 8: Comprehensive Insights and Summary

Detailed analysis of wicket patterns with actionable insights.

In [66]:

total_wickets = len(data[data['wicket'] == 1])
total_matches = data['match_id'].nunique()
avg_wickets_per_match = total_wickets / total_matches

print(f"\n1. OVERALL WICKETS STATISTICS:")
print(f"   - Total Wickets: {total_wickets}")
print(f"   - Total Matches: {total_matches}")
print(f"   - Average Wickets per Match: {avg_wickets_per_match:.2f}")
print(f"   - Total Deliveries: {len(data)}")
print(f"   - Wicket Rate: {(total_wickets/len(data)*100):.2f}%")

print(f"\n2. TOP 5 STADIUMS BY WICKETS:")
for idx, row in wickets_by_stadium.head(5).iterrows():
    print(f"   {row['Stadium']}: {int(row['Total_Wickets'])} wickets ({row['Avg_Wickets_Per_Match']:.2f} avg)")

first_inns_wickets = len(first_innings[first_innings['wicket'] == 1])
second_inns_wickets = len(second_innings[second_innings['wicket'] == 1])

print(f"\n3. INNINGS COMPARISON:")
print(f"   - 1st Innings Wickets: {first_inns_wickets} ({first_inns_wickets/total_wickets*100:.1f}%)")
print(f"   - 2nd Innings Wickets: {second_inns_wickets} ({second_inns_wickets/total_wickets*100:.1f}%)")
print(f"   - Difference: {abs(first_inns_wickets-second_inns_wickets)} wickets")



1. OVERALL WICKETS STATISTICS:
   - Total Wickets: 12950
   - Total Matches: 1095
   - Average Wickets per Match: 11.83
   - Total Deliveries: 260920
   - Wicket Rate: 4.96%

2. TOP 5 STADIUMS BY WICKETS:
   Wankhede Stadium: 1422 wickets (12.05 avg)
   Eden Gardens: 1071 wickets (11.52 avg)
   M. A. Chidambaram Stadium: 923 wickets (12.14 avg)
   M. Chinnaswamy Stadium: 922 wickets (11.52 avg)
   Feroz Shah Kotla: 697 wickets (11.62 avg)

3. INNINGS COMPARISON:
   - 1st Innings Wickets: 6695 (51.7%)
   - 2nd Innings Wickets: 6228 (48.1%)
   - Difference: 467 wickets


In [ ]:

# Compute wickets by city for plotting
wickets_by_city = data[data['wicket'] == 1].groupby('city').agg({
    'wicket': 'sum',
    'match_id': 'nunique'
}).reset_index()
wickets_by_city.columns = ['City', 'Total_Wickets', 'Matches']
wickets_by_city['Avg_Per_Match'] = (wickets_by_city['Total_Wickets'] / 
                                         wickets_by_city['Matches']).round(2)
wickets_by_city = wickets_by_city.sort_values('Total_Wickets', ascending=False)
print("Wickets by City (Top 10):")
print(wickets_by_city.head(10).to_string())

In [68]:

fig = px.bar(wickets_by_city.head(8), x='City', y='Total_Wickets',
             title='Total Wickets by City (Top 8)',
             labels={'Total_Wickets': 'Total Wickets'},
             color='Avg_Per_Match',
             color_continuous_scale='Turbo',
             text='Total_Wickets')
fig.update_layout(height=500, template='plotly_white')
fig.show()

NameError: name 'wickets_by_city' is not defined